# Validating regulatory edges with Perturb-seq

scCAFM infers a gene regulatory network from **non-targeting control cells**. In this network, `Gene1` is a transcription factor (TF), `Gene2` is a possible target gene, and the score describes the predicted strength of that relationship.

Perturb-seq provides an independent way to examine these predictions. If a predicted TF-to-target edge is biologically meaningful, perturbing the TF may change the target gene's expression. We compare target expression in TF-perturbed cells with target expression in held-out non-targeting cells using the **Wasserstein distance**. A larger distance means that the two expression distributions are more different.

This comparison is useful evidence, but it is not proof of a direct interaction. A perturbation can have indirect or off-target effects, and the distance does not describe whether expression increased or decreased.

## 1. Set up the tutorial

The model files are read from `assets/`. The K562 file under `tutorial_data/perturbseq_edge_validation/` contains raw counts after only basic cell and gene filtering. All preprocessing needed for scCAFM is performed explicitly below and does not modify the stored file.

In [ ]:
from pathlib import Path
import warnings

from IPython.display import display
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split

warnings.filterwarnings(
    "ignore",
    message="Mismatch dtype between input and weight",
)

from sccafm import (
    GRNInferencer,
    ScPreprocessor,
    evaluate_perturbseq_grn,
    load_vocab_json,
    resolve_model_assets,
)


REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

MODEL_SOURCE = REPO_ROOT / "assets"
DATA_PATH = (
    REPO_ROOT
    / "tutorial_data"
    / "perturbseq_edge_validation"
    / "K562.h5ad"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the raw-count K562 dataset at: {DATA_PATH}"
    )

assets = resolve_model_assets(MODEL_SOURCE)
token_dict = load_vocab_json(assets.vocab)
human_tfs = pd.read_csv(assets.human_tfs)
print("Tutorial files are ready.")

## 2. Examine the K562 data

K562 is a human myeloid leukaemia cell line with many measured gene perturbations. The tutorial file keeps the broad raw-count matrix rather than a small perturbation-target panel. This lets scCAFM choose an informative gene set while the held-out raw counts remain available for perturbation validation.

In [ ]:
raw = sc.read_h5ad(DATA_PATH, backed="r")

required_obs = {"gene", "gene_id", "species", "disease"}
required_var = {"gene_name"}
missing_obs = required_obs.difference(raw.obs.columns)
missing_var = required_var.difference(raw.var.columns)
if missing_obs or missing_var:
    raise KeyError(
        "The K562 file is missing required metadata: "
        + ", ".join(sorted(missing_obs | missing_var))
    )

perturbation_counts = raw.obs["gene"].astype(str).value_counts()
dataset_overview = pd.DataFrame(
    [
        {
            "Dataset": "K562",
            "Species": raw.obs["species"].astype(str).iloc[0],
            "Expression scale": "raw counts",
            "Cells": raw.n_obs,
            "Genes": raw.n_vars,
            "Non-targeting controls": perturbation_counts.get(
                "non-targeting", 0
            ),
            "Measured perturbations": (
                raw.obs["gene"].astype(str).nunique() - 1
            ),
        }
    ]
)
raw.file.close()

display(
    dataset_overview.style.format(
        {
            "Cells": "{:,.0f}",
            "Genes": "{:,.0f}",
            "Non-targeting controls": "{:,.0f}",
            "Measured perturbations": "{:,.0f}",
        }
    )
)

## 3. Prepare the expression data

We split non-targeting cells reproducibly: 80% are used to infer the pooled GRN and 20% are held out for validation. TF perturbations with more than 100 cells are eligible for validation.

`ScPreprocessor` performs model preprocessing only on the inference controls. It normalizes each cell to a total of `10000`, applies `log1p`, retains all available catalogue TFs, and selects 500 additional highly variable genes. The validation cells remain as raw counts, so Wasserstein distances are calculated on the original count scale.

In [ ]:
raw = sc.read_h5ad(DATA_PATH, backed="r")
obs = raw.obs.copy()
measured_symbols = set(
    raw.var["gene_name"].astype(str).str.upper()
)
vocabulary_symbols = set(
    token_dict["gene_symbol"].dropna().astype(str).str.upper()
)
catalog_tfs = set(
    human_tfs["TF"].dropna().astype(str).str.upper()
)

perturbation_counts = obs["gene"].astype(str).value_counts()
eligible_perturbed_tfs = sorted(
    gene
    for gene, count in perturbation_counts.items()
    if gene != "non-targeting"
    and count > 100
    and gene.upper() in measured_symbols
    and gene.upper() in vocabulary_symbols
    and gene.upper() in catalog_tfs
)
available_catalog_tfs = sorted(
    measured_symbols & vocabulary_symbols & catalog_tfs
)

control_indices = np.flatnonzero(
    obs["gene"].astype(str).to_numpy() == "non-targeting"
)
inference_indices, validation_control_indices = train_test_split(
    control_indices,
    test_size=0.2,
    random_state=0,
)
perturbed_indices = np.flatnonzero(
    obs["gene"].astype(str).isin(eligible_perturbed_tfs).to_numpy()
)
validation_indices = np.concatenate(
    [np.sort(validation_control_indices), np.sort(perturbed_indices)]
)

inference_raw = raw[np.sort(inference_indices), :].to_memory()
validation_adata = raw[validation_indices, :].to_memory()
raw.file.close()

preprocessor = ScPreprocessor(
    min_genes=200,
    min_cells=3,
    max_pct_counts_mt=20.0,
    target_sum=10000,
    log1p=True,
    n_top_genes=500,
    hvg_flavor="seurat",
    subset_hvg=True,
    remove_mito_genes=True,
    remove_ribo_genes=True,
    remove_hb_genes=True,
    token_dict=token_dict,
    gene_key="gene_name",
    preserve_gene_names=available_catalog_tfs,
    hvg_exclude_gene_names=available_catalog_tfs,
    hvg_exclude_preserved_genes=True,
    sanitize_X=True,
    inplace=False,
)
inference_adata = preprocessor(inference_raw)
# Keep the exact raw-count columns used by the model. This also removes
# ambiguous duplicate symbols from the broader source matrix.
validation_adata = validation_adata[
    :, inference_adata.var_names
].copy()

prepared_overview = pd.DataFrame(
    [
        {
            "Inference controls": inference_adata.n_obs,
            "Validation controls": len(validation_control_indices),
            "Validation perturbed cells": len(perturbed_indices),
            "Eligible perturbed TFs": len(eligible_perturbed_tfs),
            "Retained catalogue TFs": int(
                inference_adata.var.get(
                    "preserved_gene",
                    pd.Series(False, index=inference_adata.var_names),
                ).sum()
            ),
            "Additional HVGs": int(
                inference_adata.var.get(
                    "hvg_target_gene",
                    pd.Series(False, index=inference_adata.var_names),
                ).sum()
            ),
            "Model genes": inference_adata.n_vars,
        }
    ]
)
display(
    prepared_overview.style.format(
        {column: "{:,.0f}" for column in prepared_overview.columns}
    )
)

## 4. Infer a pooled GRN

The inference cells represent one non-targeting K562 population, so we average their cell-specific networks to obtain one pooled GRN. The model uses FA2 on one GPU. The returned network is unfiltered because edge selection is performed only during validation.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires one CUDA GPU.")

inferencer = GRNInferencer.from_pretrained(
    MODEL_SOURCE,
    device="cuda:0",
    attention_backend="fa2",
    max_length=2048,
    species_key="species",
    disease_key="disease",
)

pooled_grn = inferencer.infer_pooled(
    inference_adata,
    batch_size=8,
    gene_key="gene_name",
    score_threshold=None,
    top_k_edges=None,
)

pooled_overview = pd.DataFrame(
    [
        {
            "Inference cells": pooled_grn.n_cells,
            "Source TFs": len(pooled_grn.source_genes),
            "Target genes": len(pooled_grn.target_genes),
            "Candidate matrix": (
                f"{pooled_grn.shape[0]:,} × {pooled_grn.shape[1]:,}"
            ),
        }
    ]
)
display(pooled_overview)

## 5. Validate the top regulatory edges

We remove self-edges and select the 100 highest-scoring TF-to-target predictions. Perturbed cells are not used to infer or choose these edges.

For each selected edge, `evaluate_perturbseq_grn()` compares raw target-gene counts between held-out non-targeting cells and cells where the source TF was perturbed. A larger Wasserstein distance indicates a larger shift between the two raw-count distributions.

In [ ]:
TOP_K_EDGES = 100

evaluation = evaluate_perturbseq_grn(
    pooled_grn,
    validation_adata,
    perturbation_key="gene",
    control_label="non-targeting",
    top_k_edges=TOP_K_EDGES,
    gene_key="gene_name",
)
validated_edges = evaluation.to_edge_table()

validation_summary = pd.DataFrame(
    [
        {
            "Candidate edges": evaluation.n_candidates,
            "Evaluated edges": evaluation.n_evaluated_edges,
            "Perturbed TFs": evaluation.n_perturbed_tfs,
            "Validation controls": evaluation.n_control_cells,
            "Validation perturbed cells": evaluation.n_perturbed_cells,
            "Mean raw-count distance": (
                evaluation.mean_wasserstein_distance
            ),
            "Median raw-count distance": (
                evaluation.median_wasserstein_distance
            ),
        }
    ]
)
display(
    validation_summary.style.format(
        {
            "Candidate edges": "{:,.0f}",
            "Evaluated edges": "{:,.0f}",
            "Perturbed TFs": "{:,.0f}",
            "Validation controls": "{:,.0f}",
            "Validation perturbed cells": "{:,.0f}",
            "Mean raw-count distance": "{:.4f}",
            "Median raw-count distance": "{:.4f}",
        }
    )
)

display(
    validated_edges.head(10).style.format(
        {
            "rank": "{:,.0f}",
            "score": "{:.4f}",
            "n_control": "{:,.0f}",
            "n_perturbed": "{:,.0f}",
            "wasserstein_distance": "{:.4f}",
        }
    )
)

## 6. Optional: save the validation table

The cell below is disabled by default. When enabled, it saves all 100 evaluated edges with these columns:

```text
rank,Gene1,Gene2,score,n_control,n_perturbed,wasserstein_distance
```

You may change `TOP_K_EDGES` before evaluation to examine a larger or smaller ranked set. A larger set includes weaker model predictions and takes longer to validate.

In [ ]:
SAVE_VALIDATED_EDGES = False

if SAVE_VALIDATED_EDGES:
    output_dir = REPO_ROOT / "results" / "perturbseq_edge_validation"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"K562_top{TOP_K_EDGES}_validated_edges.csv"
    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite the existing file: {output_path}"
        )
    validated_edges.to_csv(output_path, index=False)
    print(f"Saved validated edges to {output_path}")
else:
    print(
        "CSV export is disabled. Set SAVE_VALIDATED_EDGES = True "
        "to save the table."
    )

## What you learned

`pooled_grn` is inferred from preprocessed non-targeting K562 controls. `validated_edges` contains its top 100 non-self predictions together with Wasserstein distances calculated from held-out raw counts.

The scCAFM score ranks predicted regulatory edges, while the Wasserstein distance summarizes the observed expression shift after perturbing the source TF. A large distance supports a perturbation response but does not prove a direct interaction.